In [0]:

import gc
from pyspark.sql import functions as F
 

TRAIN_PATH    = "/Volumes/workspace/default/raw_datasets/train_delta"
TEST_PATH     = "/Volumes/workspace/default/raw_datasets/test_delta"
PIPELINE_PATH = "/Volumes/workspace/default/raw_datasets/churn_pipeline"
 

train_df = spark.read.format("delta").load(TRAIN_PATH)
test_df  = spark.read.format("delta").load(TEST_PATH)
 

categorical_cols = [
    "gender", "Partner", "Dependents", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling",
    "PaymentMethod"
]
 
train_cnt = train_df.count()
test_cnt  = test_df.count()
total     = train_cnt + test_cnt
 
print("="*50)
print("  PHASE 4 SESSION — DATA LOADED FROM DELTA")
print("="*50)
print(f"  Train set : {train_cnt:,} rows ({train_cnt/total*100:.0f}%)")
print(f"  Test set  : {test_cnt:,}  rows ({test_cnt/total*100:.0f}%)")
print(f"  Columns   : {train_df.columns}")
print(f"\n  Features column   : {'features' in train_df.columns}")
print(f"  Label column      : {'label' in train_df.columns}")
 
# Verify feature vector dimension
sample_vec = train_df.select("features").first()[0]
print(f"  Feature vector    : {len(sample_vec)} dimensions")
 
print("\n  categorical_cols  : loaded ✓")
print(f"  ({len(categorical_cols)} columns for feature importance labels)")
print("\n Ready — proceed to Phase 4 Cell 1: Imports")

In [0]:
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import mlflow
import mlflow.spark
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore")
 
VOLUME_PATH = "/Volumes/workspace/default/raw_datasets"
mlflow.set_experiment("/churn_retention_experiment")
 
print("="*45)
print("  PHASE 4: MODEL TRAINING")
print("="*45)
print(f"Train : {train_df.count():,} rows")
print(f"Test  : {test_df.count():,} rows")
print(f"MLflow experiment : /churn_retention_experiment")


In [0]:
auc_eval = BinaryClassificationEvaluator(
    labelCol   = "label",
    metricName = "areaUnderROC"   # best metric for imbalanced data
)
pr_eval = BinaryClassificationEvaluator(
    labelCol   = "label",
    metricName = "areaUnderPR"
)
acc_eval = MulticlassClassificationEvaluator(
    labelCol      = "label",
    predictionCol = "prediction",
    metricName    = "accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol      = "label",
    predictionCol = "prediction",
    metricName    = "f1"
)
 
print("Evaluators ready: AUC-ROC, AUC-PR, Accuracy, F1")

In [0]:
def evaluate_model(name, predictions):
    """Print clean evaluation report + confusion matrix."""
    auc = auc_eval.evaluate(predictions)
    pr  = pr_eval.evaluate(predictions)
    acc = acc_eval.evaluate(predictions)
    f1  = f1_eval.evaluate(predictions)
 
    cm_pdf = predictions.groupBy("label", "prediction").count().toPandas()
 
    def safe_count(l, p):
        q = cm_pdf.query(f"label=={l} and prediction=={p}.0")
        return int(q["count"].sum()) if not q.empty else 0
 
    tp = safe_count(1, 1)
    tn = safe_count(0, 0)
    fp = safe_count(0, 1)
    fn = safe_count(1, 0)
 
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
 
    print(f"\n{'='*52}")
    print(f"  {name}")
    print(f"{'='*52}")
    print(f"  AUC-ROC   : {auc:.4f}  ← main metric (target > 0.80)")
    print(f"  AUC-PR    : {pr:.4f}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"              Predicted 0   Predicted 1")
    print(f"  Actual 0  :    {tn:>6}        {fp:>6}  ← false alarms")
    print(f"  Actual 1  :    {fn:>6}        {tp:>6}  ← caught churners")
    print(f"\n  Caught churners : {tp}/{tp+fn} ({tp/(tp+fn)*100:.1f}%)")
    print(f"  Missed churners : {fn}  ← keep this LOW for retention")
    print(f"{'='*52}")
 
    return {
        "model": name, "AUC": auc, "PR": pr,
        "Accuracy": acc, "F1": f1,
        "Precision": precision, "Recall": recall,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn
    }
 

In [0]:
print("\nTraining Logistic Regression...")
 
lr = LogisticRegression(
    featuresCol     = "features",
    labelCol        = "label",
    maxIter         = 100,
    regParam        = 0.01,
    elasticNetParam = 0.0
)
 
with mlflow.start_run(run_name="logistic_regression"):
    lr_model   = lr.fit(train_df)
    lr_preds   = lr_model.transform(test_df)
    lr_results = evaluate_model("Logistic Regression", lr_preds)
 
    mlflow.log_param("model_type",  "LogisticRegression")
    mlflow.log_param("maxIter",     100)
    mlflow.log_param("regParam",    0.01)
    mlflow.log_metric("AUC",        lr_results["AUC"])
    mlflow.log_metric("F1",         lr_results["F1"])
    mlflow.log_metric("Accuracy",   lr_results["Accuracy"])
    mlflow.log_metric("Precision",  lr_results["Precision"])
    mlflow.log_metric("Recall",     lr_results["Recall"])
 
    mlflow.spark.log_model(lr_model, "lr_model")
    lr_run_id = mlflow.active_run().info.run_id
 
print(f"\nLR Run ID : {lr_run_id}")
print("Model trained and logged successfully.")

In [0]:
print("\nTraining Random Forest (100 trees)...")
 
rf = RandomForestClassifier(
    featuresCol           = "features",
    labelCol              = "label",
    numTrees              = 100,
    maxDepth              = 5,
    minInstancesPerNode   = 5,
    featureSubsetStrategy = "sqrt",
    seed                  = 42
)
 
with mlflow.start_run(run_name="random_forest"):
    rf_model   = rf.fit(train_df)
    rf_preds   = rf_model.transform(test_df)
    rf_results = evaluate_model("Random Forest", rf_preds)
 
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("numTrees",   100)
    mlflow.log_param("maxDepth",   5)
    mlflow.log_metric("AUC",       rf_results["AUC"])
    mlflow.log_metric("F1",        rf_results["F1"])
    mlflow.log_metric("Accuracy",  rf_results["Accuracy"])
    mlflow.log_metric("Precision", rf_results["Precision"])
    mlflow.log_metric("Recall",    rf_results["Recall"])
 
    mlflow.spark.log_model(rf_model, "rf_model")
    rf_run_id = mlflow.active_run().info.run_id
 
print(f"\nRF Run ID : {rf_run_id}")

In [0]:
print("\nTraining Gradient Boosted Trees...")
 
gbt = GBTClassifier(
    featuresCol = "features",
    labelCol    = "label",
    maxIter     = 50,
    maxDepth    = 4,
    stepSize    = 0.1,
    seed        = 42
)
 
with mlflow.start_run(run_name="gradient_boosted_trees"):
    gbt_model   = gbt.fit(train_df)
    gbt_preds   = gbt_model.transform(test_df)
    gbt_results = evaluate_model("Gradient Boosted Trees", gbt_preds)
 
    mlflow.log_param("model_type", "GBTClassifier")
    mlflow.log_param("maxIter",    50)
    mlflow.log_param("maxDepth",   4)
    mlflow.log_param("stepSize",   0.1)
    mlflow.log_metric("AUC",       gbt_results["AUC"])
    mlflow.log_metric("F1",        gbt_results["F1"])
    mlflow.log_metric("Accuracy",  gbt_results["Accuracy"])
    mlflow.log_metric("Precision", gbt_results["Precision"])
    mlflow.log_metric("Recall",    gbt_results["Recall"])
 
    mlflow.spark.log_model(gbt_model, "gbt_model")
    gbt_run_id = mlflow.active_run().info.run_id
 
print(f"\nGBT Run ID : {gbt_run_id}")


In [0]:
results_df = pd.DataFrame([lr_results, rf_results, gbt_results])
results_df = results_df.set_index("model")
 
print("\n" + "="*62)
print("  MODEL COMPARISON SUMMARY")
print("="*62)
print(results_df[["AUC","F1","Accuracy","Precision","Recall"]].to_string())
print("="*62)
best = results_df["AUC"].idxmax()
print(f"\nBest model by AUC-ROC : {best} ({results_df.loc[best,'AUC']:.4f})")
print(f"\nMLflow Run IDs:")
print(f"  LR  : {lr_run_id}")
print(f"  RF  : {rf_run_id}")
print(f"  GBT : {gbt_run_id}")
print("View all runs: Experiments → /churn_retention_experiment")

In [0]:
print("\nTop 15 most important features:")
 
encoded_names = [c + "_enc" for c in categorical_cols]
feature_names = encoded_names + [
    "SeniorCitizen_scaled", "tenure_scaled",
    "MonthlyCharges_scaled", "TotalCharges_scaled"
]
importances = rf_model.featureImportances.toArray()
min_len     = min(len(feature_names), len(importances))
 
feat_imp = pd.DataFrame({
    "feature"    : feature_names[:min_len],
    "importance" : importances[:min_len]
}).sort_values("importance", ascending=False)
 
display(feat_imp.head(15))

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Model Performance Comparison", fontsize=14, fontweight="bold")
 
metrics = ["AUC", "F1", "Accuracy", "Precision", "Recall"]
models  = results_df.index.tolist()
x       = np.arange(len(metrics))
width   = 0.25
colors  = ["#2196F3", "#4CAF50", "#FF9800"]
 
for i, (model_name, color) in enumerate(zip(models, colors)):
    vals = [results_df.loc[model_name, m] for m in metrics]
    bars = axes[0].bar(x + i*width, vals, width,
                       label=model_name, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.005,
                     f"{val:.2f}", ha="center", fontsize=8)
 
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1.15)
axes[0].set_title("All metrics by model")
axes[0].legend(fontsize=9)
 
top10 = feat_imp.head(10)
axes[1].barh(top10["feature"][::-1], top10["importance"][::-1],
             color="#4CAF50", alpha=0.85)
axes[1].set_title("Top 10 Feature Importances (Random Forest)")
axes[1].set_xlabel("Importance score")
 
plt.tight_layout()
plt.savefig(f"{VOLUME_PATH}/model_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to volume.")

In [0]:
extract_prob = F.udf(lambda v: float(v[1]), DoubleType())
 
rf_with_prob = rf_preds.withColumn(
    "churn_prob", extract_prob(F.col("probability"))
)
rf_adjusted = rf_with_prob.withColumn(
    "prediction_adj",
    F.when(F.col("churn_prob") >= 0.4, 1.0).otherwise(0.0)
)
 
adj_f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction_adj", metricName="f1"
).evaluate(rf_adjusted)
 
print(f"\nThreshold comparison (Random Forest):")
print(f"  Default 0.5  → F1: {rf_results['F1']:.4f}")
print(f"  Lowered 0.4  → F1: {adj_f1:.4f}")
print("\nLower threshold = catches more churners (preferred for retention)")

In [0]:
MODEL_PATH = f"{VOLUME_PATH}/churn_rf_model"
rf_model.write().overwrite().save(MODEL_PATH)
 
print(f"\nRandom Forest model saved to : {MODEL_PATH}")
print("Reload in Phase 5 with:")
print("  from pyspark.ml.classification import RandomForestClassificationModel")
print(f"  rf_model = RandomForestClassificationModel.load('{MODEL_PATH}')")
print("\n PHASE 4 COMPLETE — proceed to Phase 5: Risk Scoring & Priority Tiers")